# AIML Lab Assignment #7 — Overfitting Control in Neural Networks
**Course:** CSET301 | **Dataset:** MNIST | **Semester:** 4th Even 2025

---
### Objectives
- Identify overfitting in neural networks
- Apply techniques: L2 Regularization, Dropout, Early Stopping, Data Augmentation
- Evaluate and compare model performance

## Step 0: Install & Import Libraries

In [ ]:
# Install required packages (run once)
# !pip install tensorflow scikit-learn matplotlib seaborn numpy

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("TensorFlow version:", tf.__version__)
print("All libraries imported successfully!")

---
## Task 1: Data Exploration
Load the MNIST dataset, inspect shape, classes, and visualize samples.

In [ ]:
# Load MNIST dataset
(X_train_full, y_train_full), (X_test_orig, y_test_orig) = keras.datasets.mnist.load_data()

print("=== MNIST Dataset Info ===")
print(f"Training data shape : {X_train_full.shape}")
print(f"Training labels     : {y_train_full.shape}")
print(f"Test data shape     : {X_test_orig.shape}")
print(f"Test labels         : {y_test_orig.shape}")
print(f"Pixel value range   : {X_train_full.min()} - {X_train_full.max()}")
print(f"Number of classes   : {len(np.unique(y_train_full))}")
print(f"Classes             : {np.unique(y_train_full)}")

In [ ]:
# Visualize sample images from each class
fig, axes = plt.subplots(2, 10, figsize=(18, 4))
fig.suptitle('MNIST Sample Images (one per class)', fontsize=14, fontweight='bold')

for digit in range(10):
    idx = np.where(y_train_full == digit)[0][0]
    axes[0, digit].imshow(X_train_full[idx], cmap='gray')
    axes[0, digit].set_title(f'Digit: {digit}', fontsize=9)
    axes[0, digit].axis('off')

    idx2 = np.where(y_train_full == digit)[0][1]
    axes[1, digit].imshow(X_train_full[idx2], cmap='gray')
    axes[1, digit].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

unique, counts = np.unique(y_train_full, return_counts=True)
axes[0].bar(unique, counts, color='steelblue', edgecolor='black')
axes[0].set_title('Class Distribution — Training Set', fontweight='bold')
axes[0].set_xlabel('Digit Class')
axes[0].set_ylabel('Count')
axes[0].set_xticks(unique)
for i, v in zip(unique, counts):
    axes[0].text(i, v + 50, str(v), ha='center', va='bottom', fontsize=8)

unique_t, counts_t = np.unique(y_test_orig, return_counts=True)
axes[1].bar(unique_t, counts_t, color='coral', edgecolor='black')
axes[1].set_title('Class Distribution — Test Set', fontweight='bold')
axes[1].set_xlabel('Digit Class')
axes[1].set_ylabel('Count')
axes[1].set_xticks(unique_t)
for i, v in zip(unique_t, counts_t):
    axes[1].text(i, v + 5, str(v), ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()
print("Dataset is well-balanced across all 10 classes.")

---
## Task 2: Data Preparation
Normalize pixel values and split into train / validation / test sets (60-20-20).

In [ ]:
# ---- Normalize pixel values to [0, 1] ----
X_all = X_train_full.astype('float32') / 255.0
X_test_norm = X_test_orig.astype('float32') / 255.0

y_all = y_train_full
y_test = y_test_orig

# ---- Combine all data and split 60-20-20 ----
X_combined = np.concatenate([X_all, X_test_norm], axis=0)
y_combined = np.concatenate([y_all, y_test], axis=0)

# First split: 80% train+val, 20% test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_combined, y_combined, test_size=0.20, random_state=42, stratify=y_combined
)
# Second split: from remaining 80%, take 25% as val → 60% train, 20% val
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval
)

# ---- Reshape for CNN (add channel dimension) ----
X_train_cnn = X_train.reshape(-1, 28, 28, 1)
X_val_cnn   = X_val.reshape(-1, 28, 28, 1)
X_test_cnn  = X_test.reshape(-1, 28, 28, 1)

# ---- Flatten for MLP ----
X_train_flat = X_train.reshape(-1, 784)
X_val_flat   = X_val.reshape(-1, 784)
X_test_flat  = X_test.reshape(-1, 784)

print("=== Data Split Summary ===")
print(f"Train      : {X_train.shape[0]:>6} samples  ({X_train.shape[0]/len(X_combined)*100:.1f}%)")
print(f"Validation : {X_val.shape[0]:>6} samples  ({X_val.shape[0]/len(X_combined)*100:.1f}%)")
print(f"Test       : {X_test.shape[0]:>6} samples  ({X_test.shape[0]/len(X_combined)*100:.1f}%)")
print(f"\nCNN input shape : {X_train_cnn.shape[1:]}")
print(f"MLP input shape : {X_train_flat.shape[1:]}")
print(f"Pixel range after normalization: {X_train.min():.2f} – {X_train.max():.2f}")

In [ ]:
# ---- (Optional) Data Augmentation Setup ----
datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=False  # digits are not symmetric
)
datagen.fit(X_train_cnn)

# Show augmented samples
sample_img = X_train_cnn[:1]
fig, axes = plt.subplots(1, 8, figsize=(16, 2))
fig.suptitle('Data Augmentation — Augmented versions of one sample', fontweight='bold')
axes[0].imshow(sample_img[0, :, :, 0], cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')
aug_iter = datagen.flow(sample_img, batch_size=1)
for i in range(1, 8):
    aug_img = next(aug_iter)[0]
    axes[i].imshow(aug_img[:, :, 0], cmap='gray')
    axes[i].set_title(f'Aug {i}')
    axes[i].axis('off')
plt.tight_layout()
plt.show()

---
## Task 3: Baseline Model (No Overfitting Control)
Train a CNN without any regularization to deliberately observe overfitting.

In [ ]:
def build_baseline_cnn():
    """Overfit-prone CNN — no regularization, deep enough to memorize."""
    model = keras.Sequential([
        layers.Input(shape=(28, 28, 1)),

        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.Dense(256, activation='relu'),
        layers.Dense(10, activation='softmax')
    ], name='Baseline_CNN')
    return model

baseline_model = build_baseline_cnn()
baseline_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
baseline_model.summary()

In [ ]:
BASELINE_EPOCHS = 30

history_baseline = baseline_model.fit(
    X_train_cnn, y_train,
    epochs=BASELINE_EPOCHS,
    batch_size=64,
    validation_data=(X_val_cnn, y_val),
    verbose=1
)

baseline_test_loss, baseline_test_acc = baseline_model.evaluate(X_test_cnn, y_test, verbose=0)
print(f"\nBaseline Test Accuracy : {baseline_test_acc*100:.2f}%")
print(f"Baseline Test Loss     : {baseline_test_loss:.4f}")

In [ ]:
def plot_history(history, title='Training History', ax=None):
    """Plot training vs validation accuracy and loss."""
    if ax is None:
        fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    epochs = range(1, len(history.history['accuracy']) + 1)

    ax[0].plot(epochs, history.history['accuracy'],    'b-o', markersize=3, label='Train Accuracy')
    ax[0].plot(epochs, history.history['val_accuracy'],'r-o', markersize=3, label='Val Accuracy')
    ax[0].set_title(f'{title} — Accuracy', fontweight='bold')
    ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Accuracy')
    ax[0].legend(); ax[0].grid(alpha=0.3)

    ax[1].plot(epochs, history.history['loss'],    'b-o', markersize=3, label='Train Loss')
    ax[1].plot(epochs, history.history['val_loss'],'r-o', markersize=3, label='Val Loss')
    ax[1].set_title(f'{title} — Loss', fontweight='bold')
    ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('Loss')
    ax[1].legend(); ax[1].grid(alpha=0.3)

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
plot_history(history_baseline, title='Baseline CNN (Overfitting Observed)', ax=ax)
plt.tight_layout()
plt.show()

# Measure overfitting gap
train_acc_final = history_baseline.history['accuracy'][-1]
val_acc_final   = history_baseline.history['val_accuracy'][-1]
print(f"\nOverfitting Gap (Train Acc - Val Acc) = {(train_acc_final - val_acc_final)*100:.2f}%")
print("→ Large positive gap confirms overfitting.")

---
## Task 4: Overfitting Control Techniques
### 4.1 — L2 Regularization (Weight Decay)

In [ ]:
def build_l2_cnn(l2_lambda=0.001):
    """CNN with L2 regularization on Dense layers."""
    reg = regularizers.l2(l2_lambda)
    model = keras.Sequential([
        layers.Input(shape=(28, 28, 1)),

        layers.Conv2D(32, (3, 3), activation='relu', padding='same', kernel_regularizer=reg),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', kernel_regularizer=reg),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation='relu', padding='same', kernel_regularizer=reg),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same', kernel_regularizer=reg),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dense(512, activation='relu', kernel_regularizer=reg),
        layers.Dense(256, activation='relu', kernel_regularizer=reg),
        layers.Dense(10, activation='softmax')
    ], name='L2_CNN')
    return model

l2_model = build_l2_cnn(l2_lambda=0.001)
l2_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_l2 = l2_model.fit(
    X_train_cnn, y_train,
    epochs=30,
    batch_size=64,
    validation_data=(X_val_cnn, y_val),
    verbose=1
)

l2_test_loss, l2_test_acc = l2_model.evaluate(X_test_cnn, y_test, verbose=0)
print(f"\nL2 Regularization Test Accuracy : {l2_test_acc*100:.2f}%")

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
plot_history(history_l2, title='L2 Regularization CNN', ax=ax)
plt.tight_layout()
plt.show()

### 4.2 — Dropout Regularization

In [ ]:
def build_dropout_cnn(dropout_rate=0.3):
    """CNN with Dropout layers."""
    model = keras.Sequential([
        layers.Input(shape=(28, 28, 1)),

        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(dropout_rate / 2),   # lighter dropout after conv

        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(dropout_rate / 2),

        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(dropout_rate),       # heavier dropout on dense
        layers.Dense(256, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(10, activation='softmax')
    ], name=f'Dropout_{dropout_rate}_CNN')
    return model

dropout_model = build_dropout_cnn(dropout_rate=0.3)
dropout_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_dropout = dropout_model.fit(
    X_train_cnn, y_train,
    epochs=30,
    batch_size=64,
    validation_data=(X_val_cnn, y_val),
    verbose=1
)

dropout_test_loss, dropout_test_acc = dropout_model.evaluate(X_test_cnn, y_test, verbose=0)
print(f"\nDropout (0.3) Test Accuracy : {dropout_test_acc*100:.2f}%")

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
plot_history(history_dropout, title='Dropout CNN (rate=0.3)', ax=ax)
plt.tight_layout()
plt.show()

### 4.3 — Early Stopping

In [ ]:
# Use same baseline architecture but with EarlyStopping callback
early_stop_model = build_baseline_cnn()
early_stop_model._name = 'EarlyStopping_CNN'
early_stop_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stopping_cb = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

history_es = early_stop_model.fit(
    X_train_cnn, y_train,
    epochs=50,           # allow up to 50 — ES will stop it early
    batch_size=64,
    validation_data=(X_val_cnn, y_val),
    callbacks=[early_stopping_cb],
    verbose=1
)

es_test_loss, es_test_acc = early_stop_model.evaluate(X_test_cnn, y_test, verbose=0)
print(f"\nEarly Stopping Test Accuracy : {es_test_acc*100:.2f}%")
print(f"Stopped at epoch             : {len(history_es.history['loss'])}")

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
plot_history(history_es, title='Early Stopping CNN', ax=ax)
plt.tight_layout()
plt.show()

### 4.4 — Data Augmentation

In [ ]:
augmented_model = build_baseline_cnn()
augmented_model._name = 'DataAugmentation_CNN'
augmented_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Create augmented data generator
aug_datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1
)
aug_datagen.fit(X_train_cnn)
train_aug_gen = aug_datagen.flow(X_train_cnn, y_train, batch_size=64)

steps_per_epoch = len(X_train_cnn) // 64

history_aug = augmented_model.fit(
    train_aug_gen,
    steps_per_epoch=steps_per_epoch,
    epochs=30,
    validation_data=(X_val_cnn, y_val),
    verbose=1
)

aug_test_loss, aug_test_acc = augmented_model.evaluate(X_test_cnn, y_test, verbose=0)
print(f"\nData Augmentation Test Accuracy : {aug_test_acc*100:.2f}%")

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
plot_history(history_aug, title='Data Augmentation CNN', ax=ax)
plt.tight_layout()
plt.show()

---
## Task 5: Hyperparameter Tuning
Experiment with different Dropout rates, L2 lambdas, batch sizes, and learning rates.

In [ ]:
# ---- Experiment: Different Dropout Rates ----
dropout_rates = [0.2, 0.3, 0.5]
dropout_results = {}

for rate in dropout_rates:
    print(f"\n--- Training with Dropout rate = {rate} ---")
    m = build_dropout_cnn(dropout_rate=rate)
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    h = m.fit(
        X_train_cnn, y_train,
        epochs=20,
        batch_size=64,
        validation_data=(X_val_cnn, y_val),
        verbose=0
    )
    _, test_acc = m.evaluate(X_test_cnn, y_test, verbose=0)
    gap = h.history['accuracy'][-1] - h.history['val_accuracy'][-1]
    dropout_results[rate] = {'history': h, 'test_acc': test_acc, 'overfit_gap': gap}
    print(f"  Test Acc: {test_acc*100:.2f}% | Overfit Gap: {gap*100:.2f}%")

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
colors = ['green', 'blue', 'red']
for rate, col in zip(dropout_rates, colors):
    h = dropout_results[rate]['history']
    e = range(1, len(h.history['val_accuracy']) + 1)
    axes[0].plot(e, h.history['val_accuracy'], color=col, label=f'Dropout={rate}')
    axes[1].plot(e, h.history['val_loss'],     color=col, label=f'Dropout={rate}')

axes[0].set_title('Val Accuracy — Different Dropout Rates', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Val Accuracy')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].set_title('Val Loss — Different Dropout Rates', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ---- Experiment: Different L2 Lambda Values ----
l2_lambdas = [0.0001, 0.001, 0.01]
l2_results = {}

for lam in l2_lambdas:
    print(f"\n--- Training with L2 lambda = {lam} ---")
    m = build_l2_cnn(l2_lambda=lam)
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    h = m.fit(
        X_train_cnn, y_train,
        epochs=20,
        batch_size=64,
        validation_data=(X_val_cnn, y_val),
        verbose=0
    )
    _, test_acc = m.evaluate(X_test_cnn, y_test, verbose=0)
    gap = h.history['accuracy'][-1] - h.history['val_accuracy'][-1]
    l2_results[lam] = {'history': h, 'test_acc': test_acc, 'overfit_gap': gap}
    print(f"  Test Acc: {test_acc*100:.2f}% | Overfit Gap: {gap*100:.2f}%")

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
colors = ['green', 'blue', 'red']
for lam, col in zip(l2_lambdas, colors):
    h = l2_results[lam]['history']
    e = range(1, len(h.history['val_accuracy']) + 1)
    axes[0].plot(e, h.history['val_accuracy'], color=col, label=f'λ={lam}')
    axes[1].plot(e, h.history['val_loss'],     color=col, label=f'λ={lam}')

axes[0].set_title('Val Accuracy — Different L2 Lambdas', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Val Accuracy')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].set_title('Val Loss — Different L2 Lambdas', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ---- Experiment: Different Batch Sizes & Learning Rates ----
configs = [
    {'batch_size': 32,  'lr': 0.001},
    {'batch_size': 64,  'lr': 0.001},
    {'batch_size': 128, 'lr': 0.001},
    {'batch_size': 64,  'lr': 0.0001},
]
config_results = []

for cfg in configs:
    print(f"\n--- batch={cfg['batch_size']}, lr={cfg['lr']} ---")
    m = build_dropout_cnn(dropout_rate=0.3)
    m.compile(optimizer=keras.optimizers.Adam(cfg['lr']),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
    h = m.fit(
        X_train_cnn, y_train,
        epochs=15,
        batch_size=cfg['batch_size'],
        validation_data=(X_val_cnn, y_val),
        verbose=0
    )
    _, test_acc = m.evaluate(X_test_cnn, y_test, verbose=0)
    config_results.append({'config': cfg, 'test_acc': test_acc,
                           'val_acc': max(h.history['val_accuracy'])})
    print(f"  Test Acc: {test_acc*100:.2f}%")

print("\n=== Hyperparameter Tuning Summary ===")
print(f"{'Batch':>6} {'LR':>8} {'Test Acc':>10} {'Best Val Acc':>14}")
print("-" * 42)
for r in config_results:
    print(f"{r['config']['batch_size']:>6} {r['config']['lr']:>8} {r['test_acc']*100:>9.2f}% {r['val_acc']*100:>13.2f}%")

---
## Additional Task: Combined Techniques
Dropout + Early Stopping + Data Augmentation together for best performance.

In [ ]:
def build_combined_cnn(dropout_rate=0.3, l2_lambda=0.0005):
    """Best model: L2 + Dropout combined."""
    reg = regularizers.l2(l2_lambda)
    model = keras.Sequential([
        layers.Input(shape=(28, 28, 1)),

        layers.Conv2D(32, (3, 3), activation='relu', padding='same', kernel_regularizer=reg),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', kernel_regularizer=reg),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.15),

        layers.Conv2D(64, (3, 3), activation='relu', padding='same', kernel_regularizer=reg),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same', kernel_regularizer=reg),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.15),

        layers.Flatten(),
        layers.Dense(512, activation='relu', kernel_regularizer=reg),
        layers.Dropout(dropout_rate),
        layers.Dense(256, activation='relu', kernel_regularizer=reg),
        layers.Dropout(dropout_rate),
        layers.Dense(10, activation='softmax')
    ], name='Combined_L2_Dropout_CNN')
    return model

best_model = build_combined_cnn(dropout_rate=0.3, l2_lambda=0.0005)
best_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Data Augmentation
best_datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1
)
best_datagen.fit(X_train_cnn)
best_gen = best_datagen.flow(X_train_cnn, y_train, batch_size=64)

# Early Stopping
es_cb = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1)

history_combined = best_model.fit(
    best_gen,
    steps_per_epoch=len(X_train_cnn) // 64,
    epochs=50,
    validation_data=(X_val_cnn, y_val),
    callbacks=[es_cb],
    verbose=1
)

combined_loss, combined_acc = best_model.evaluate(X_test_cnn, y_test, verbose=0)
print(f"\n Combined Model Test Accuracy : {combined_acc*100:.2f}%")
print(f"Combined Model Test Loss     : {combined_loss:.4f}")

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
plot_history(history_combined, title='Combined: L2 + Dropout + Aug + EarlyStopping', ax=ax)
plt.tight_layout()
plt.show()

---
## Task 6: Performance Evaluation
### 6.1 — Compare All Models

In [ ]:
# Evaluate all models on test set
models_info = [
    ('Baseline (No Control)',          baseline_model,    history_baseline),
    ('L2 Regularization',              l2_model,          history_l2),
    ('Dropout (rate=0.3)',             dropout_model,     history_dropout),
    ('Early Stopping',                 early_stop_model,  history_es),
    ('Data Augmentation',              augmented_model,   history_aug),
    ('Combined (Best Model)',          best_model,        history_combined),
]

results = []
for name, model, history in models_info:
    _, test_acc = model.evaluate(X_test_cnn, y_test, verbose=0)
    train_acc_f = history.history['accuracy'][-1]
    val_acc_f   = history.history['val_accuracy'][-1]
    gap = train_acc_f - val_acc_f
    results.append({'Model': name,
                    'Train Acc': f'{train_acc_f*100:.2f}%',
                    'Val Acc':   f'{val_acc_f*100:.2f}%',
                    'Test Acc':  f'{test_acc*100:.2f}%',
                    'Overfit Gap': f'{gap*100:.2f}%'})

print(f"{'Model':<30} {'Train Acc':>10} {'Val Acc':>10} {'Test Acc':>10} {'Overfit Gap':>12}")
print("-" * 75)
for r in results:
    print(f"{r['Model']:<30} {r['Train Acc']:>10} {r['Val Acc']:>10} {r['Test Acc']:>10} {r['Overfit Gap']:>12}")

In [ ]:
# Visual comparison bar chart
model_names = [r['Model'] for r in results]
test_accs   = [float(r['Test Acc'].strip('%')) for r in results]
overfit_gaps= [float(r['Overfit Gap'].strip('%')) for r in results]

x = np.arange(len(model_names))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

bars1 = axes[0].bar(x, test_accs, color='steelblue', edgecolor='black')
axes[0].set_title('Test Accuracy per Model', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_names, rotation=25, ha='right', fontsize=8)
axes[0].set_ylim([min(test_accs) - 2, 100])
axes[0].grid(axis='y', alpha=0.3)
for bar, val in zip(bars1, test_accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=8)

colors_gap = ['red' if g > 2 else 'green' for g in overfit_gaps]
bars2 = axes[1].bar(x, overfit_gaps, color=colors_gap, edgecolor='black')
axes[1].set_title('Overfitting Gap (Train - Val Accuracy) — Lower is Better',
                   fontweight='bold', fontsize=11)
axes[1].set_ylabel('Gap (%)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(model_names, rotation=25, ha='right', fontsize=8)
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].grid(axis='y', alpha=0.3)
for bar, val in zip(bars2, overfit_gaps):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.05 if val >= 0 else bar.get_height() - 0.5,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

### 6.2 — Classification Report (Best Model)

In [ ]:
# Predict on test set using best model
y_pred_prob = best_model.predict(X_test_cnn, verbose=0)
y_pred      = np.argmax(y_pred_prob, axis=1)

print("=" * 55)
print("   Classification Report — Combined Best Model")
print("=" * 55)
print(classification_report(y_test, y_pred,
                             target_names=[f'Digit {i}' for i in range(10)]))

### 6.3 — Confusion Matrix (Best Model)

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10), ax=ax,
            linewidths=0.5, linecolor='gray')
ax.set_title('Confusion Matrix — Combined Best Model', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.show()

# Per-class accuracy
per_class = cm.diagonal() / cm.sum(axis=1) * 100
print("\nPer-class accuracy:")
for i, acc in enumerate(per_class):
    print(f"  Digit {i}: {acc:.2f}%")

### 6.4 — All Learning Curves Side by Side

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 14))
fig.suptitle('Learning Curves — All Models', fontsize=16, fontweight='bold')

histories_to_plot = [
    (history_baseline, 'Baseline (No Control)'),
    (history_l2,       'L2 Regularization'),
    (history_dropout,  'Dropout (rate=0.3)'),
    (history_es,       'Early Stopping'),
    (history_aug,      'Data Augmentation'),
    (history_combined, 'Combined Best Model'),
]

for idx, (hist, title) in enumerate(histories_to_plot):
    row, col = divmod(idx, 2)
    ax = axes[row][col]
    e = range(1, len(hist.history['accuracy']) + 1)
    ax.plot(e, hist.history['accuracy'],     'b-', label='Train Acc', linewidth=1.5)
    ax.plot(e, hist.history['val_accuracy'], 'r--', label='Val Acc',  linewidth=1.5)
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_ylim([0.85, 1.01])

plt.tight_layout()
plt.show()

---
## Discussion: Bias-Variance Trade-off

| Technique | Bias | Variance | Effect |
|---|---|---|---|
| **No Regularization** | Low | High | Overfits — memorizes training data |
| **L2 Regularization** | ↑ Slight | ↓ Reduces | Penalizes large weights; generalization improves |
| **Dropout** | ↑ Slight | ↓ Reduces | Forces redundant learning; acts like ensemble |
| **Early Stopping** | ↑ Slight | ↓ Reduces | Stops before model memorizes; simple and effective |
| **Data Augmentation** | ≈ Same | ↓ Reduces | More data diversity; model sees more real-world variation |
| **Combined** | Balanced | Minimized | Best generalization — all techniques complement each other |

> **Key Insight:** Too strong regularization pushes the model toward underfitting (high bias). The sweet spot is where validation accuracy peaks — achieved best by combining mild versions of multiple techniques.


---
## Final Summary

In [ ]:
print("=" * 65)
print("          AIML Lab Week 7 — Final Results Summary")
print("=" * 65)
print(f"{'Model':<30} {'Test Accuracy':>14} {'Overfit Gap':>12}")
print("-" * 60)
for r in results:
    marker = " ★" if r['Model'] == 'Combined (Best Model)' else ""
    print(f"{r['Model']:<30} {r['Test Acc']:>14} {r['Overfit Gap']:>12}{marker}")
print("-" * 60)
print("\n★ Best Model: L2 + Dropout + Data Augmentation + Early Stopping")
print("\nConclusions:")
print("  1. Baseline model clearly overfits (large train-val gap)")
print("  2. Each individual technique reduces overfitting")
print("  3. Combining techniques gives best results")
print("  4. Early stopping is computationally cheapest anti-overfitting method")
print("  5. Data augmentation is most effective when labeled data is scarce")